# Cognitive-Motor Command Response Analysis (Adapted from Claassen 2019)

In [1]:
# %pip install matplotlib
# %pip install PyQt5
# %pip install pyyaml
# %pip install mne
# %pip install numpy
# %pip install pandas
# %pip uninstall -y scikit-learn
# %pip install scikit-learn
# %pip install --upgrade jupyter ipywidgets

import matplotlib.pyplot as plt
import matplotlib
import PyQt5

# Use matplotlib Qt5Agg backend - best choice for MNE-Python interactive plotting functions
matplotlib.use('Qt5Agg')

import yaml
import sys
import os

# Add the parent directory of 'data' to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from preprocessing.eeg_loader import load_eeg, get_eeg_timestamps, load_stimulus, detect_signal_start
config_file_path = '../configs/claassen_cfg.yml'  # Replace with the actual path to your config file
with open(config_file_path, 'r') as file:
    config = yaml.safe_load(file)

from tqdm import tqdm_notebook
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, cross_val_score, LeaveOneGroupOut

import mne
import numpy as np
import pandas as pd

from mne.time_frequency import psd_array_multitaper
from mne.decoding import LinearModel, get_coef

### 1. Preprocessing Raw Data

In [2]:
# eeg_path = r"/Users/joobeejung/EEG_DATA/X~ X_ef7f7805-3781-4f1e-8134-8d5b57750330.EDF"
# eeg_path = r"/Users/joobeejung/EEG_DATA/X~ X_86b44d5d-5036-4748-b666-ceff710a1e8d.EDF" #CON001
# eeg_path = r"/Users/joobeejung/EEG_DATA/CON002_20240924.EDF"
# eeg_path = r"/Users/joobeejung/EEG_DATA/CON003_clipped.EDF"

subject = "CON001b" #CON001a, CON001b, CON002, CON003, CON004
eeg_path = config.get(f"{subject}_path", "")  # Remove extra quotes

if eeg_path is None:
    print(f"Error: No path found for subject {subject}")
else:
    print(f"EEG Path for {subject}: {eeg_path}")

# Check if file exists
if not os.path.exists(eeg_path):
    raise FileNotFoundError(f"File not found: {eeg_path}")

#---Read raw without removing channels---
# # Read raw data
raw = mne.io.read_raw_edf(eeg_path, preload=True)

# # Band-pass filter between 1-30 Hz over the readings
fmin, fmax = 1, 30 # Hz
raw.filter(l_freq=fmin, h_freq=fmax)


EEG Path for CON001b: /Users/joobeejung/EEG_DATA/X~ X_86b44d5d-5036-4748-b666-ceff710a1e8d.EDF
Extracting EDF parameters from /Users/joobeejung/EEG_DATA/X~ X_86b44d5d-5036-4748-b666-ceff710a1e8d.EDF...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2052095  =      0.000 ...  4007.998 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1691 samples (3.303 s)



/var/folders/x9/r75gjp_s3cb8kqm4k8t7t_nm0000gn/T/ipykernel_8570/568277118.py:20: RuntimeWarning: Omitted 5 annotation(s) that were outside data range.
  raw = mne.io.read_raw_edf(eeg_path, preload=True)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.3s


<RawEDF | X~ X_86b44d5d-5036-4748-b666-ceff710a1e8d.EDF, 50 x 2052096 (4008.0 s), ~782.9 MiB, data loaded>

In [3]:
#---Read raw & removing channels---
fname, raw, dc_channel = load_eeg(eeg_path, config, subject)

# raw.set_eeg_reference('average')

Extracting EDF parameters from /Users/joobeejung/EEG_DATA/X~ X_86b44d5d-5036-4748-b666-ceff710a1e8d.EDF...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2052095  =      0.000 ...  4007.998 secs...
channels: ['C3', 'C4', 'O1', 'O2', 'FT9', 'FT10', 'Cz', 'F3', 'F4', 'F7', 'F8', 'Fz', 'Fp1', 'Fp2', 'Fpz', 'P3', 'P4', 'Pz', 'T7', 'T8', 'P7', 'P8', 'IO1', 'IO2', 'EMG1', 'EMG2', 'ECGL', 'ECGR', 'LAT1', 'LAT2', 'RAT1', 'RAT2', 'RESP', 'ABD', 'FLOW', 'SNORE', 'DIF5', 'DIF6', 'POS', 'DC2', 'DC3', 'DC4', 'DC5', 'DC6', 'DC7', 'DC8', 'DC9', 'DC10', 'OSAT', 'PR']
Resampling data to 512 Hz
Sampling frequency of the instance is already 512.0, returning unmodified.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 

/Users/joobeejung/eeg-auditory-stimulus/preprocessing/eeg_loader.py:12: RuntimeWarning: Omitted 5 annotation(s) that were outside data range.
  raw = mne.io.read_raw_edf(eeg_path, preload=True)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.2s
/Users/joobeejung/eeg-auditory-stimulus/preprocessing/eeg_loader.py:60: RuntimeWarning: The unit for channel(s) DC10, DC2, DC3, DC4, DC5, DC6, DC7, DC8, DC9 has changed from V to NA.
  raw.set_channel_types(channel_type_mapping)


DC channels with signals: ['DC5']


In [5]:
raw.plot()

<MNEBrowseFigure size 1600x1600 with 4 Axes>

Channels marked as bad:
none


### 3. Reading events and segmenting trials into epochs

In [6]:
# start_time, end_time = get_eeg_timestamps(raw, subject)
start_time, end_time = get_eeg_timestamps(raw)

df, patient_id = load_stimulus(config["event_full_path"], start_time, end_time)
print(patient_id)

patient_trial = df.loc[(df['patient_id'] == patient_id) & (df['trial_type'].isin(config['trial_type']))]
print(patient_trial)

def trial_start_sec(row):
    return (row['start_time']-start_time).total_seconds()
def trial_end_sec(row):
    return (row['end_time']-start_time).total_seconds()

patient_trial['start_sec'] = patient_trial.apply(trial_start_sec, axis=1)
patient_trial['end_sec'] = patient_trial.apply(trial_end_sec, axis=1)
df = patient_trial
expanded_rows = []


CON001b
    patient_id        date trial_type sentences                start_time  \
188    CON001b  2024-09-17       lcmd        [] 2024-09-17 22:45:26+00:00   
192    CON001b  2024-09-17       lcmd        [] 2024-09-17 22:50:03+00:00   
193    CON001b  2024-09-17       rcmd        [] 2024-09-17 22:53:32+00:00   
206    CON001b  2024-09-17       rcmd        [] 2024-09-17 23:00:28+00:00   
235    CON001b  2024-09-17       rcmd        [] 2024-09-17 23:12:31+00:00   
236    CON001b  2024-09-17       lcmd        [] 2024-09-17 23:16:00+00:00   

                     end_time    duration  
188 2024-09-17 22:48:54+00:00  207.764528  
192 2024-09-17 22:53:30+00:00  207.908227  
193 2024-09-17 22:57:00+00:00  207.312183  
206 2024-09-17 23:03:55+00:00  207.093676  
235 2024-09-17 23:15:58+00:00  207.255298  
236 2024-09-17 23:19:28+00:00  207.899271  


/var/folders/x9/r75gjp_s3cb8kqm4k8t7t_nm0000gn/T/ipykernel_8570/1321512825.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  patient_trial['start_sec'] = patient_trial.apply(trial_start_sec, axis=1)
/var/folders/x9/r75gjp_s3cb8kqm4k8t7t_nm0000gn/T/ipykernel_8570/1321512825.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  patient_trial['end_sec'] = patient_trial.apply(trial_end_sec, axis=1)


In [7]:
for _, row in df.iterrows():
    trial_start_sec = row['start_sec']
    signal_start_time = trial_start_sec

    for i in range(8):
        signal_start_sample, signal_start_time = detect_signal_start(raw, trial_start_sec, dc_channel)
        new_row = row.copy()
        new_row['start_sample'] = signal_start_sample
        new_row['start_sec'] = signal_start_time
        if i % 2 == 0:
            new_row['trial_type'] = f"{row['trial_type']}-keep"
        else:
            new_row['trial_type'] = f"{row['trial_type']}-stop"
        expanded_rows.append(new_row)
        trial_start_sec = signal_start_time + 10

new_df = pd.DataFrame(expanded_rows)
previous_values = [0] * len(new_df)

instruction_dict = {
    'rcmd-keep': 1,
    'rcmd-stop': 2,
    'lcmd-keep': 1,
    'lcmd-stop': 2
}

# Map trial types to event IDs
new_df['event_id'] = new_df['trial_type'].map(instruction_dict)
event_ids = new_df['event_id'].tolist()  # Removed addition of a final event

# # Combine results into the instruction array
instructions = np.column_stack([new_df['start_sample'], previous_values, event_ids])


In [11]:
# Each 10s following a given instruction is split into 5 epochs (each of 2s)
n_epo_segments = 5
n_samples = raw.info['sfreq']*10/n_epo_segments

events = list()
events_info = list()

# For each instruction
for instr_id, (onset, _, code) in enumerate(instructions):
    # Generate 5 epochs
    for repeat in range(n_epo_segments):
        # event = [onset + repeat * n_samples + config['sfreq'], 0, code]
        event = [onset + repeat * n_samples + 2.3*config['sfreq'], 0, code]

        # Store into new event array
        events.append(event)
        events_info.append(instr_id)
events = np.array(events, int)

# Add information
metadata = pd.DataFrame(dict(
    time_sample=events[:, 0], # time sample in the EEG file
    id=events[:, 2], # the unique code of the epoch
    move=(events[:, 2] % 2) == 1, # whether the code corresponds to a 'move' trial
    instr=events_info, # the instruction from which the epoch comes
    trial=np.array(events_info)//2, # trial number: there are two instructions per trial
))

# There are 8 trials per block
metadata['block'] = metadata['trial'] // 8

In [12]:
metadata.head(10)

,time_sample,id,move,instr,trial,block
0,467264,1,True,0,0,0
1,468288,1,True,0,0,0
2,469312,1,True,0,0,0
3,470336,1,True,0,0,0
4,471360,1,True,0,0,0
5,473826,2,False,1,0,0
6,474850,2,False,1,0,0
7,475874,2,False,1,0,0
8,476898,2,False,1,0,0
9,477922,2,False,1,0,0


In [13]:
np.set_printoptions(threshold=np.inf)

In [14]:
# Plot trial/instruction/epoch structure for clarity
plt.scatter(instructions[:, 0], instructions[:, 2],
            color=['g', 'r'] * (len(instructions) // 2),  # Ensure the size matches
            marker='s',
            label='Instruction offset')
plt.scatter(events[:,0], events[:,2]+.1,
            color=np.ravel([['g'] * 5 + ['r'] * 5] * (len(instructions)//2)),
            label='Epoch onset')

plt.ylabel('Instructions')
plt.yticks([1,2])
plt.gca().set_yticklabels(['Keep moving...', 'Stop moving...'])
# plt.xlim(400000, 700000)
plt.legend()
plt.grid(True)
plt.show()

In [15]:
use_ch = config.get(f"{subject}_channels")  # Append "_channels" to match config keys
use_ch = [ch for ch in use_ch if ch != "DC7"]  # Exclude "DC7"

if use_ch is None:
    use_ch = config.get("channels")
    print(f"Error: No channels found for subject {subject}")
else:
    print(f"EEG channels for {subject}: {use_ch}")
    
if raw.info['dig'] is None:
    print("No digitization data found. Assigning standard montage.")
    raw.set_montage(mne.channels.make_standard_montage("standard_1020"))

picks = mne.pick_types(raw.info, include=use_ch)
epochs = mne.Epochs(
    raw,
    tmin=0, tmax=2,
    picks=picks,
    events=events,
    metadata=metadata,
    preload=True,
    proj=False,
    baseline=None
)


EEG channels for CON001b: ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'Pz', 'P3', 'P4', 'T5', 'T6', 'O1', 'O2']
No digitization data found. Assigning standard montage.
Adding metadata with 6 columns
240 matching events found
No baseline correction applied
Using data from preloaded Raw for 240 events and 1025 original time points ...
0 bad epochs dropped


In [ ]:
import pycsd

epochs.info['description'] = 'standard/1020'
epochs = mne.preprocessing.compute_current_source_density(epochs)

epochs.plot(scalings='auto', n_epochs=3)

Fitted sphere radius:         95.4 mm
Origin head coordinates:      -1.0 15.6 45.4 mm
Origin device coordinates:    -1.0 15.6 45.4 mm


<MNEBrowseFigure size 1600x1600 with 4 Axes>

Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


## 3. Performing Power Spectral Density (PSD) Analysis

In [17]:
data = epochs.get_data()  # Shape: (n_epochs, n_channels, n_times)
n_epochs, n_channels, n_times = data.shape

# Initialize a container for storing PSDs
psds_all_epochs = []

# Loop through each epoch
for epoch_data in data:  # Shape: (n_channels, n_times)
    psds, freqs = psd_array_multitaper(
        epoch_data, sfreq=512, fmin=1, fmax=30, verbose=False
    )
    psds_all_epochs.append(psds)  # Append PSD for this epoch

# Convert to NumPy array for easier manipulation
psds_all_epochs = np.array(psds_all_epochs)  # Shape: (n_epochs, n_channels, n_freqs)

print(psds_all_epochs.shape)  # Check shape


(240, 19, 58)


In [18]:
n_epochs, n_chans, n_freqs = psds_all_epochs.shape

#Frequency bands of interest in Hz
bands = ((1,3), (4,7), (8,13), (14,30))

# Setup X array average PSD within a given frequency band
psd_data = np.zeros((n_epochs, n_chans, len(bands)))
for ii, (fmin, fmax) in enumerate(bands):
    #Find frequencies
    freq_index = np.where(np.logical_and(freqs >= fmin, freqs <= fmax))[0]

    psd_data[:, :, ii] = psds_all_epochs[:, :, freq_index].mean(2)

# Vectorize PSD
psd_data = psd_data.reshape(n_epochs, n_chans * len(bands))


## 4. Defining Cross Validation

In [19]:
# Define cross validation
cv = LeaveOneGroupOut()

fig, axes = plt.subplots(4, 1, figsize=[14,9])
axes = iter(axes)

for split, (train, test) in enumerate(cv.split(
    X = psd_data,
    y = epochs.metadata['move'],
    groups = epochs.metadata['trial'],
)):
    if split >= 3 and split != 8:
        continue
    ax = next(axes)

    ax.scatter(epochs.metadata['time_sample'].values[train], 1 - epochs.metadata['move'].values[train], color='k', label='train')
    ax.scatter(epochs.metadata['time_sample'].values[test], 1 - epochs.metadata['move'].values[test], edgecolor='b', color='w', label='test')
    ax.set_title('CV Split #%i' % (split+1))
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['"Keep moving..."', '"Stop moving..."'])
    ax.set_ylabel('Instruction')
    ax.set_xlabel('Time')
    ax.set_xticks([])
    # ax.set_xlim(0, 100000) # zoom for clarity
    ax.legend()

fig.tight_layout()
fig.show()


## 5. Defining classifier (SVM)

In [20]:
clf = make_pipeline(
    StandardScaler(),
    SVC(kernel='linear', probability=True)
)

## 6. Decoding performance over time

In [21]:
y_pred = cross_val_predict(
    clf,
    X = psd_data,
    y = epochs.metadata['move'],
    method = 'predict_proba',
    cv = cv,
    groups = epochs.metadata['trial']
)

epochs.metadata['y_pred'] = y_pred[:, 1]



In [22]:
# Average the proba over the 3 blocks to obtain temporal pattern
proba = np.mean([block['y_pred'] for _, block in epochs.metadata.groupby('block')], axis=0)

# Plot the proba
fig, ax = plt.subplots(figsize=(12,4))
ax.set_ylabel('P ("Keep moving ...")')
ax.set_xlabel('Epoch number')
ax.plot(proba, marker='o', linestyle='-', linewidth=3, markersize=6)
ax.set_ylim([0,1])
plt.axhline(0.5, linestyle=':', color='k', label='Chance')

for x in np.arange(-0.5, 79, 10):
    plt.axvline(x, color='g', label='Keep moving ...' if x < 4 else None)
    plt.axvline(x + 5, color='r', label='Stop moving ...' if x < 4 else None)

plt.legend(loc='lower right', framealpha=1.)
plt.title("Average predicted probability of 'keep moving ...' across the three blocks") 

plt.show()


In [23]:
use_ch = config.get(f"{subject}_channels")  # Append "_channels" to match config keys
use_ch = [ch for ch in use_ch if ch != "DC7"]  # Exclude "DC7"

if use_ch is None:
    print(f"Error: No channels found for subject {subject}")
else:
    print(f"EEG channels for {subject}: {use_ch}")
raw.pick(use_ch)


EEG channels for CON001b: ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'Pz', 'P3', 'P4', 'T5', 'T6', 'O1', 'O2']


<RawEDF | X~ X_86b44d5d-5036-4748-b666-ceff710a1e8d.EDF, 19 x 2052096 (4008.0 s), ~297.5 MiB, data loaded>

## 5. Topo Map

In [25]:
# Define the classifier and stode spatial patterns
# To plot the SVM patterns, it is necessary to compute the data
# covariance (Haufe et al Neuroimage 2014).
# Spatial patterns are automatically stored by MNE LinearModel.
clf = make_pipeline(
StandardScaler(), # z-score to center data
LinearModel(LinearSVC())) # Linear SVM augmented with an automatic storing of spatial patterns
# fit classifier
clf.fit(X=psd_data,
y=epochs.metadata['move'])
# Unscale the spatial patterns before plotting
patterns = get_coef(clf, 'patterns_', inverse_transform=True)
# In our study, the SVM is trained on all frequencies simultanouesly
# we thus pull the corresponding spatial topographies apart
n_elect = int(patterns.shape[0] / len(bands))
spatial_pattern = patterns.reshape(n_elect, len(bands))
# Plot
# montage = mne.channels.read_montage('standard/1020')
montage = mne.channels.make_standard_montage('standard_1020')

# The paper used this code but this may not match actual EEG data
# raw.set_montage(montage)
# pos = mne.channels.layout.find_layout(raw.info['chs']).pos

# Get electrode positions directly from montage - This way is more reliable for real data, ensures correct channel order and locations
pos = np.array([raw.info['chs'][i]['loc'][:2] for i in range(len(raw.info['chs'])) if raw.info['chs'][i]['kind'] == mne.io.constants.FIFF.FIFFV_EEG_CH])

fig, axes = plt.subplots(1, len(bands),figsize=(12,3))
fig.suptitle("EEG Spatial Patterns Across Frequency Bands", fontsize=14)
fig.tight_layout()

for idx, (band, sp, ax) in enumerate(zip(bands, spatial_pattern.T, axes)):
    scale = np.percentile(np.abs(sp), 99)
    im, _ = mne.viz.plot_topomap(sp, pos, vlim=(-scale,+scale), cmap='RdBu_r', axes=ax, show=False)
    # plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Weight (Arbitrary Units)", fontsize=8)  # Update with correct units if known

    ax.set_title('%i - %i Hz' % band)

plt.show()

## 6. Computing cross-validated AUC scores

In [26]:
clf = make_pipeline(
    StandardScaler(),
    LinearSVC()
)

scores = cross_val_score(
    estimator=clf,
    X=psd_data,
    y=epochs.metadata['move'],
    scoring='roc_auc',
    cv=cv,
    groups=epochs.metadata['trial']
)

mean_score = scores.mean(0)
print('Mean scores across split: AUC=%.3f' % mean_score)

Mean scores across split: AUC=0.785


## 7. Diagnosis of cognitive motor dissociation (CMD)

### 7.1 Performing permutation test

In [27]:
permutation_scores = []
n_permutations = 500
order = np.arange(len(epochs))

# %pip install -U ipywidgets
# %pip install jupyter
# %pip install --upgrade jupyter ipywidgets

from tqdm.notebook import tqdm  
for _ in tqdm(range(n_permutations)):  
    # Shuffle order
    np.random.shuffle(order)

    # Compute score with similar parameters
    permutation_score = cross_val_score(
        estimator=clf,
        X=psd_data,
        y=epochs.metadata['move'].values[order],
        scoring='roc_auc',
        cv=cv,
        groups=epochs.metadata['trial'].values[order],
        n_jobs=-1,
    )

    # Store results
    permutation_scores.append(permutation_score.mean(0))

  0%|          | 0/500 [00:00<?, ?it/s]

/Users/joobeejung/eeg-auditory-stimulus/pycsd_env/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/joobeejung/eeg-auditory-stimulus/pycsd_env/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/joobeejung/eeg-auditory-stimulus/pycsd_env/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/joobeejung/eeg-auditory-stimulus/pycsd_env/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/joobeejung/eeg-auditory-stimulus/pycsd_env/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the

In [28]:
observed_score = cross_val_score(
    estimator=clf,
    X=psd_data,
    y=epochs.metadata['move'].values,
    scoring='roc_auc',
    cv=cv,
    groups=epochs.metadata['trial'].values,
    n_jobs=-1,
).mean(0)

p_value = np.mean(np.array(permutation_scores) >= observed_score)
print(p_value)

import matplotlib.pyplot as plt

plt.hist(permutation_scores, bins=30, alpha=0.7, label='Permutation scores')
plt.axvline(observed_score, color='red', linestyle='--', label='Observed score')
plt.xlabel('ROC AUC Score')
plt.ylabel('Frequency')
plt.legend()
plt.title('Permutation Test Null Distribution')
plt.show()


0.0


In [ ]:
# %pip install seaborn

import seaborn as sns
# The p-value is computed from the number of permutations which
# leads to a higher score than the one obtained without permutation
# p = n_higher + 1 / (n_permutation + 1)
#
# (Ojala M GG. Journal of Machine Learning Research. 2010).
n_higher = sum([s >= scores.mean(0) for s in permutation_scores])
pvalue = (n_higher + 1.) / (n_permutations + 1.)
print("Empirical AUC = %.2f +/-%.2f" % (scores.mean(0), scores.std(0)))
print("Shuffle AUC = %.2f" % np.mean(permutation_scores, 0))
print("p-value = %.4f" % pvalue)
# plot permutation and empirical distributions
sns.kdeplot(permutation_scores, label='permutation scores')
sns.kdeplot(scores)
plt.axvline(.5, linestyle='--', label='theoretical chance')
plt.axvline(scores.mean(), color='orange', label='mean score')
plt.scatter(scores, 6. + np.random.randn(len(scores))/10., color='orange', s=5, label='split score')
plt.xlim(-.1, 1.1)
plt.legend()
plt.xlabel('AUC Score')
plt.ylabel('Probability')
plt.yticks([])
plt.show()

Empirical AUC = 0.79 +/-0.18
Shuffle AUC = 0.50
p-value = 0.0020


: 